# 🏭 Solution IA : Contrôle Intelligent du Chauffage des Gaz
## Élimination de la Pollution Visuelle des Cheminées (Panache Blanc)

---

### 🎯 Contexte
Les cheminées des unités de production d'acide phosphorique et DAP émettent des **panaches blancs visibles** constitués essentiellement d'**air saturé en vapeur d'eau** (aucun impact polluant réel).  
La solution : **chauffer les gaz en amont** pour empêcher la condensation — mais uniquement quand c'est nécessaire, grâce à l'IA.

### 📋 Objectifs du Notebook
1. Générer un dataset météo + opératoire réaliste
2. Modéliser la visibilité du panache (classification)
3. Prédire la puissance de chauffage optimale (régression)
4. Construire un système de décision intelligent
5. Quantifier les économies d'énergie réalisées

---

## 📦 0. Installation des Dépendances

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost plotly shap lightgbm -q
print('✅ Librairies installées!')

## 📚 1. Importation des Librairies

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    mean_squared_error, mean_absolute_error, r2_score
)
import xgboost as xgb
import pickle

np.random.seed(42)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid', palette='muted')

print('✅ Librairies importées!')

## 🌦️ 2. Génération du Dataset

Simulation de **3 ans de données horaires** (26 280 heures) incluant :
- Paramètres météo (humidité, température, vent, pression)
- Paramètres des gaz de cheminée (température, débit, humidité)
- Visibilité réelle du panache (labellisée)
- Puissance de chauffage utilisée
- Consommation énergétique

In [ ]:
def dew_point(T, H):
    """Calcul du point de rosée (formule Magnus)"""
    a, b = 17.27, 237.7
    alpha = (a * T) / (b + T) + np.log(np.maximum(H, 1) / 100)
    return (b * alpha) / (a - alpha)


def generate_panache_dataset(n_hours=26280, seed=42):
    """
    Génère un dataset réaliste pour le contrôle intelligent
    du chauffage des gaz de cheminée (élimination panache visuel).
    
    Localisation : Gabès, Tunisie (climat méditerranéen semi-aride)
    Unité        : Cheminée acide phosphorique + DAP
    """
    np.random.seed(seed)
    
    dates = pd.date_range(start='2021-01-01', periods=n_hours, freq='H')
    hour    = dates.hour
    month   = dates.month
    dow     = dates.dayofweek
    
    # ─── Météo Gabès ────────────────────────────────────────────
    # Température ambiante (°C) — saisonnalité méditerranéenne
    T_base   = 20 + 10 * np.sin(2*np.pi*(month - 4)/12)          # 10°C hiver → 30°C été
    T_diurne = 5  * np.sin(np.pi*(hour - 5)/12) * (hour>=5)*(hour<=19)
    T_amb    = T_base + T_diurne + np.random.normal(0, 1.5, n_hours)
    T_amb    = np.clip(T_amb, 2, 44)

    # Humidité relative (%) — plus humide en hiver/nuit
    H_base   = 65 - 20 * np.sin(2*np.pi*(month - 1)/12)          # 85% hiver → 45% été
    H_diurne = -15 * np.sin(np.pi*(hour - 5)/12) * (hour>=5)*(hour<=19)
    H_rel    = H_base + H_diurne + np.random.normal(0, 4, n_hours)
    H_rel    = np.clip(H_rel, 15, 100)

    # Vent (m/s)
    vitesse_vent = np.abs(np.random.weibull(1.8, n_hours) * 4)
    vitesse_vent = np.clip(vitesse_vent, 0, 18)
    direction_vent = np.random.uniform(0, 360, n_hours)

    # Pression atmosphérique (hPa)
    pression_atm = 1013 + 5*np.sin(2*np.pi*np.arange(n_hours)/8760) + np.random.normal(0, 3, n_hours)

    # Point de rosée
    T_rosee = dew_point(T_amb, H_rel)

    # Visibilité horizontale (km) — réduite par brume/brouillard
    visibilite_km = np.where(H_rel > 90, np.random.uniform(0.5, 3, n_hours),
                    np.where(H_rel > 75, np.random.uniform(3, 8, n_hours),
                    np.random.uniform(8, 25, n_hours)))

    # ─── Paramètres des gaz de cheminée ─────────────────────────
    # Débit volumique (Nm³/h) — varie avec la production
    production_factor = 0.7 + 0.3*(dow < 5) + np.random.normal(0, 0.05, n_hours)
    production_factor = np.clip(production_factor, 0.4, 1.1)

    debit_gaz = 45000 * production_factor + np.random.normal(0, 2000, n_hours)
    debit_gaz = np.clip(debit_gaz, 25000, 65000)

    # Température des gaz en sortie procédé (°C)
    T_gaz_procede = 55 + 10*production_factor + np.random.normal(0, 3, n_hours)
    T_gaz_procede = np.clip(T_gaz_procede, 42, 80)

    # Humidité des gaz (%)
    H_gaz = 88 + 6*(production_factor - 0.85) + np.random.normal(0, 2, n_hours)
    H_gaz = np.clip(H_gaz, 78, 99)

    # ─── Modèle physique de visibilité du panache ────────────────
    # Le panache est visible si les gaz se condensent au contact de l'air
    # Condition : T_mélange < T_rosée_mélange
    # Approximation : panache visible si T_gaz - T_amb < seuil(H_rel)
    delta_T_critique = np.where(
        H_rel > 80, 15,
        np.where(H_rel > 60, 25, 40)
    )
    delta_T_reel = T_gaz_procede - T_amb
    
    prob_visible = 1 / (1 + np.exp(0.3*(delta_T_reel - delta_T_critique)))
    panache_visible = (np.random.uniform(0, 1, n_hours) < prob_visible).astype(int)

    # Intensité visuelle du panache (0 = invisible, 3 = très dense)
    intensite_panache = np.zeros(n_hours)
    intensite_panache[panache_visible == 1] = np.clip(
        (H_rel[panache_visible == 1] - 50) / 17, 0.5, 3
    )
    intensite_panache = np.round(intensite_panache, 1)

    # ─── Puissance de chauffage nécessaire ──────────────────────
    # Pour rendre le panache invisible : élever T_gaz de ΔT
    # ΔT_requis = f(H_rel, T_amb, débit)
    delta_T_requis = np.where(
        (H_rel < 50) | (hour < 6) | (hour > 21), 0,
        np.where(H_rel < 65, 12,
        np.where(H_rel < 80, 25,
        np.where(T_amb > 25, 15, 45)))
    ) + np.random.normal(0, 2, n_hours)
    delta_T_requis = np.clip(delta_T_requis, 0, 60)

    # Puissance thermique (kW) = débit × Cp_air × ΔT / 3600
    # Cp_air ≈ 1.005 kJ/(kg·K), densité air ≈ 1.2 kg/m³
    puissance_kW = debit_gaz * 1.2 * 1.005 * delta_T_requis / 3600
    puissance_kW = np.clip(puissance_kW, 0, 250)

    # Stratégie sans IA : chauffage continu (puissance fixe = 120 kW)
    puissance_sans_ia = np.where(
        (hour >= 6) & (hour <= 22) & (dow < 5), 120, 60
    ).astype(float)

    # Coût énergétique (DT/h) — tarif industriel 0.18 DT/kWh
    tarif_kwh = 0.18
    cout_avec_ia    = puissance_kW * tarif_kwh
    cout_sans_ia    = puissance_sans_ia * tarif_kwh
    economie_horaire = cout_sans_ia - cout_avec_ia

    # ─── Classification de la décision IA ───────────────────────
    def classify_decision(row):
        h, hr, t, pw = row
        if h < 6 or h > 22:
            return 'nuit_inactif'
        if pw == 0:
            return 'chauffage_inutile'
        if pw < 30:
            return 'chauffage_leger'
        if pw < 80:
            return 'chauffage_modere'
        return 'chauffage_maximal'

    decision_ia = np.array([
        classify_decision(r) for r in zip(hour, H_rel, T_amb, puissance_kW)
    ])

    # ─── Assemblage ─────────────────────────────────────────────
    df = pd.DataFrame({
        'timestamp'        : dates,
        'heure'            : hour,
        'mois'             : month,
        'jour_semaine'     : dow,
        'saison'           : pd.cut(month, [0,3,6,9,12],
                                labels=['Hiver','Printemps','Été','Automne']),

        # Météo
        'T_amb_C'          : np.round(T_amb, 1),
        'humidite_rel_pct' : np.round(H_rel, 1),
        'T_rosee_C'        : np.round(T_rosee, 1),
        'vitesse_vent_ms'  : np.round(vitesse_vent, 2),
        'direction_vent_deg': np.round(direction_vent, 0),
        'pression_hpa'     : np.round(pression_atm, 1),
        'visibilite_km'    : np.round(visibilite_km, 1),

        # Gaz cheminée
        'T_gaz_procede_C'  : np.round(T_gaz_procede, 1),
        'H_gaz_pct'        : np.round(H_gaz, 1),
        'debit_gaz_nm3h'   : np.round(debit_gaz, 0),
        'delta_T_gaz_amb'  : np.round(delta_T_reel, 1),

        # Panache
        'panache_visible'  : panache_visible,
        'intensite_panache': intensite_panache,

        # Chauffage
        'delta_T_requis_C' : np.round(delta_T_requis, 1),
        'puissance_ia_kW'  : np.round(puissance_kW, 2),
        'puissance_sans_ia_kW': puissance_sans_ia,

        # Coûts
        'cout_ia_dth'      : np.round(cout_avec_ia, 4),
        'cout_sans_ia_dth' : np.round(cout_sans_ia, 4),
        'economie_dth'     : np.round(economie_horaire, 4),

        # Décision
        'decision_ia'      : decision_ia,
        'production_factor': np.round(production_factor, 3),
    })

    df.set_index('timestamp', inplace=True)
    return df


print('⏳ Génération du dataset...')
df = generate_panache_dataset(n_hours=26280)

print(f'\n✅ Dataset généré!')
print(f'   Dimensions    : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(f'   Période       : {df.index.min().date()} → {df.index.max().date()}')
print(f'   Localisation  : Gabès, Tunisie')
print(f'   Panache visible : {df["panache_visible"].mean()*100:.1f}% du temps')
print(f'   Économie potentielle : {df["economie_dth"].sum():,.0f} DT sur 3 ans')

df.head()

## 📊 3. Statistiques Descriptives

In [ ]:
print('='*65)
print('         STATISTIQUES DESCRIPTIVES DU DATASET')
print('='*65)

cols_stats = [
    'T_amb_C', 'humidite_rel_pct', 'T_rosee_C', 'vitesse_vent_ms',
    'T_gaz_procede_C', 'H_gaz_pct', 'delta_T_requis_C',
    'puissance_ia_kW', 'puissance_sans_ia_kW', 'economie_dth'
]
display(df[cols_stats].describe().round(2))

print('\n📊 Distribution des décisions IA :')
dec_counts = df['decision_ia'].value_counts()
for dec, cnt in dec_counts.items():
    pct = cnt / len(df) * 100
    print(f'   {dec:<25} : {cnt:>6,} heures ({pct:5.1f}%)')

print(f'\n💡 Panache visible    : {df["panache_visible"].sum():,} heures ({df["panache_visible"].mean()*100:.1f}%)')
print(f'   Économie totale   : {df["economie_dth"].sum():,.0f} DT sur 3 ans')
print(f'   Économie annuelle : {df["economie_dth"].sum()/3:,.0f} DT/an')

## 📈 4. Analyse Exploratoire (EDA)

In [ ]:
# ─── Figure 1 : Patterns saisonniers et journaliers ───
fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Profil journalier de visibilité du panache
hourly_vis = df.groupby('heure')['panache_visible'].mean() * 100
hourly_pow = df.groupby('heure')['puissance_ia_kW'].mean()

ax1 = axes[0, 0]
color1 = '#3B8BD4'
color2 = '#E8593C'
ln1 = ax1.fill_between(hourly_vis.index, hourly_vis.values, alpha=0.3, color=color1)
ax1.plot(hourly_vis.index, hourly_vis.values, 'o-', color=color1, linewidth=2, label='Visibilité panache (%)')
ax1.set_ylabel('Panache visible (%)', color=color1)
ax1.set_xlabel('Heure')
ax2 = ax1.twinx()
ax2.plot(hourly_pow.index, hourly_pow.values, 's--', color=color2, linewidth=2, label='Puissance IA (kW)')
ax2.set_ylabel('Puissance chauffage (kW)', color=color2)
ax1.set_title('⏰ Profil Journalier : Visibilité vs Chauffage IA', fontweight='bold')
ax1.set_xticks(range(0, 24, 2))
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

# Variation mensuelle de la puissance
monthly_pow_ia    = df.groupby('mois')['puissance_ia_kW'].mean()
monthly_pow_no_ia = df.groupby('mois')['puissance_sans_ia_kW'].mean()
months_fr = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']
x = np.arange(12)
w = 0.35
axes[0,1].bar(x - w/2, monthly_pow_no_ia.values, w, label='Sans IA (continu)', color='#e74c3c', alpha=0.7)
axes[0,1].bar(x + w/2, monthly_pow_ia.values, w, label='Avec IA (intelligent)', color='#2ecc71', alpha=0.7)
axes[0,1].set_xticks(x)
axes[0,1].set_xticklabels(months_fr, rotation=45)
axes[0,1].set_ylabel('Puissance moyenne (kW)')
axes[0,1].set_title('📅 Comparaison Mensuelle Puissance', fontweight='bold')
axes[0,1].legend()

# Relation humidité → visibilité
bins = np.arange(15, 105, 5)
df['H_bin'] = pd.cut(df['humidite_rel_pct'], bins=bins)
vis_by_H = df.groupby('H_bin')['panache_visible'].mean() * 100
centers = [(b.left + b.right) / 2 for b in vis_by_H.index]
colors_bars = ['#2ecc71' if v < 30 else '#f39c12' if v < 60 else '#e74c3c' for v in vis_by_H.values]
axes[1,0].bar(centers, vis_by_H.values, width=4.5, color=colors_bars, alpha=0.8, edgecolor='white')
axes[1,0].set_xlabel('Humidité Relative (%)')
axes[1,0].set_ylabel('Panache visible (%)')
axes[1,0].set_title('💧 Humidité vs Probabilité de Visibilité', fontweight='bold')
legend_elems = [
    mpatches.Patch(color='#2ecc71', label='< 30% visible'),
    mpatches.Patch(color='#f39c12', label='30–60% visible'),
    mpatches.Patch(color='#e74c3c', label='> 60% visible'),
]
axes[1,0].legend(handles=legend_elems, fontsize=9)

# Distribution des décisions IA
dec_labels = {'nuit_inactif': '🌙 Nuit inactif',
              'chauffage_inutile': '✅ Chauffage inutile',
              'chauffage_leger': '🟡 Léger',
              'chauffage_modere': '🟠 Modéré',
              'chauffage_maximal': '🔴 Maximal'}
dec_colors = ['#85B7EB', '#2ecc71', '#f1c40f', '#f39c12', '#e74c3c']
dec_vals = df['decision_ia'].value_counts()
labels_mapped = [dec_labels.get(k, k) for k in dec_vals.index]
axes[1,1].pie(dec_vals.values, labels=labels_mapped, autopct='%1.1f%%',
               colors=dec_colors, startangle=90, textprops={'fontsize': 10})
axes[1,1].set_title('🤖 Répartition Décisions IA', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_panache.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Graphique EDA sauvegardé.')

In [ ]:
# ─── Figure 2 : Série temporelle + Carte de chaleur ───
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Série temporelle sur 2 semaines
sample = df['2021-12-01':'2021-12-14']
axes[0].fill_between(sample.index, sample['puissance_sans_ia_kW'],
                      alpha=0.4, color='red', label='Sans IA (continu)')
axes[0].fill_between(sample.index, sample['puissance_ia_kW'],
                      alpha=0.7, color='green', label='Avec IA')
axes[0].fill_between(sample.index,
                      sample['panache_visible'] * sample['puissance_sans_ia_kW'].max(),
                      alpha=0.15, color='blue', label='Panache visible')
axes[0].set_ylabel('Puissance (kW)')
axes[0].set_title('⚡ Puissance Chauffage : Sans IA vs Avec IA (Décembre 2021)', fontweight='bold')
axes[0].legend()

# Heatmap heure × mois de visibilité
pivot = df.groupby(['mois', 'heure'])['panache_visible'].mean().unstack() * 100
sns.heatmap(pivot, ax=axes[1], cmap='RdYlGn_r', annot=False,
            xticklabels=range(0, 24), yticklabels=months_fr,
            cbar_kws={'label': 'Panache visible (%)'})
axes[1].set_xlabel('Heure')
axes[1].set_ylabel('Mois')
axes[1].set_title('🗓️ Carte de Chaleur : Visibilité Panache (Heure × Mois)', fontweight='bold')

plt.tight_layout()
plt.savefig('serie_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Matrice de corrélation ───
num_cols = [
    'T_amb_C', 'humidite_rel_pct', 'T_rosee_C', 'vitesse_vent_ms',
    'pression_hpa', 'T_gaz_procede_C', 'H_gaz_pct', 'debit_gaz_nm3h',
    'delta_T_gaz_amb', 'panache_visible', 'puissance_ia_kW'
]
fig, ax = plt.subplots(figsize=(12, 9))
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, center=0, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Corrélation'})
ax.set_title('🔗 Matrice de Corrélation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_panache.png', dpi=150, bbox_inches='tight')
plt.show()

## 🤖 5. Modèles de Machine Learning

### 5.1 Modèle 1 : Prédiction de Visibilité du Panache (Classification)

In [ ]:
features_cls = [
    'T_amb_C', 'humidite_rel_pct', 'T_rosee_C', 'vitesse_vent_ms',
    'pression_hpa', 'T_gaz_procede_C', 'H_gaz_pct', 'debit_gaz_nm3h',
    'delta_T_gaz_amb', 'heure', 'mois', 'jour_semaine', 'production_factor'
]

X_cls = df[features_cls].values
y_cls = df['panache_visible'].values

# Split temporel (80/20)
split = int(0.8 * len(X_cls))
X_tr, X_te = X_cls[:split], X_cls[split:]
y_tr, y_te = y_cls[:split], y_cls[split:]

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc  = scaler.transform(X_te)

print(f'Train: {len(X_tr):,} | Test: {len(X_te):,} | Features: {len(features_cls)}')

models_cls = {
    'Logistic Regression': LogisticRegression(max_iter=500, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=150, max_depth=14, n_jobs=-1, random_state=42),
    'XGBoost'             : xgb.XGBClassifier(n_estimators=150, max_depth=7, random_state=42, n_jobs=-1, verbosity=0),
}

results_cls = {}
print('\n⏳ Entraînement classifieurs...')
for name, model in models_cls.items():
    if name == 'Logistic Regression':
        model.fit(X_tr_sc, y_tr)
        y_pred = model.predict(X_te_sc)
    else:
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
    acc = accuracy_score(y_te, y_pred)
    results_cls[name] = {'acc': acc, 'pred': y_pred, 'model': model}
    print(f'   ✓ {name:<25} | Accuracy = {acc:.4f}')

best_cls_name = max(results_cls, key=lambda k: results_cls[k]['acc'])
print(f'\n🏆 Meilleur : {best_cls_name} (Acc={results_cls[best_cls_name]["acc"]:.4f})')
print()
print('Rapport détaillé :')
print(classification_report(y_te, results_cls[best_cls_name]['pred'],
                              target_names=['Invisible','Visible']))

In [ ]:
# Visualisation classification
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Comparaison accuracy
names = list(results_cls.keys())
accs  = [results_cls[n]['acc'] for n in names]
colors_bar = ['#2ecc71' if n == best_cls_name else '#3498db' for n in names]
axes[0].barh(names, accs, color=colors_bar, alpha=0.85)
axes[0].set_xlabel('Accuracy')
axes[0].set_title('📊 Accuracy — Prédiction Visibilité', fontweight='bold')
axes[0].set_xlim(0, 1.05)
for i, (n, v) in enumerate(zip(names, accs)):
    axes[0].text(v + 0.005, i, f'{v:.4f}', va='center')

# Matrice de confusion
cm = confusion_matrix(y_te, results_cls[best_cls_name]['pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Prédit Invisible','Prédit Visible'],
            yticklabels=['Réel Invisible','Réel Visible'], ax=axes[1])
axes[1].set_title(f'Matrice Confusion — {best_cls_name}', fontweight='bold')

plt.tight_layout()
plt.savefig('classification_panache.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Modèle 2 : Prédiction de la Puissance de Chauffage Optimale (Régression)

In [ ]:
# Uniquement les heures où le chauffage est utile
df_heat = df[(df['puissance_ia_kW'] > 0) & (df['heure'] >= 6) & (df['heure'] <= 22)].copy()
print(f'Données pour régression : {len(df_heat):,} heures (chauffage actif)')

features_reg = [
    'T_amb_C', 'humidite_rel_pct', 'T_rosee_C', 'vitesse_vent_ms',
    'T_gaz_procede_C', 'H_gaz_pct', 'debit_gaz_nm3h',
    'delta_T_gaz_amb', 'heure', 'mois', 'production_factor'
]

X_reg = df_heat[features_reg].values
y_reg = df_heat['puissance_ia_kW'].values

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(X_reg, y_reg, test_size=0.2, random_state=42)

models_reg = {
    'Ridge'              : Ridge(alpha=1.0),
    'Random Forest'      : RandomForestRegressor(n_estimators=150, max_depth=12, n_jobs=-1, random_state=42),
    'Gradient Boosting'  : GradientBoostingRegressor(n_estimators=150, max_depth=6, random_state=42),
    'XGBoost'            : xgb.XGBRegressor(n_estimators=200, max_depth=7, learning_rate=0.05,
                                              random_state=42, n_jobs=-1, verbosity=0),
}

results_reg = {}
scaler_r = StandardScaler()
X_tr_r_sc = scaler_r.fit_transform(X_tr_r)
X_te_r_sc  = scaler_r.transform(X_te_r)

print('\n⏳ Entraînement régresseurs...')
for name, model in models_reg.items():
    if name == 'Ridge':
        model.fit(X_tr_r_sc, y_tr_r)
        y_pred = model.predict(X_te_r_sc)
    else:
        model.fit(X_tr_r, y_tr_r)
        y_pred = model.predict(X_te_r)
    rmse = np.sqrt(mean_squared_error(y_te_r, y_pred))
    r2   = r2_score(y_te_r, y_pred)
    results_reg[name] = {'rmse': rmse, 'r2': r2, 'pred': y_pred, 'model': model}
    print(f'   ✓ {name:<22} | RMSE={rmse:.2f} kW | R²={r2:.4f}')

best_reg_name = max(results_reg, key=lambda k: results_reg[k]['r2'])
print(f'\n🏆 Meilleur : {best_reg_name} (R²={results_reg[best_reg_name]["r2"]:.4f})')

In [ ]:
# Visualisation régression
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² comparaison
names_r = list(results_reg.keys())
r2s     = [results_reg[n]['r2'] for n in names_r]
colors_r = ['#2ecc71' if n == best_reg_name else '#3498db' for n in names_r]
axes[0].barh(names_r, r2s, color=colors_r, alpha=0.85)
axes[0].set_xlabel('Score R²')
axes[0].set_title('📊 R² — Prédiction Puissance Chauffage', fontweight='bold')
for i, v in enumerate(r2s):
    axes[0].text(v + 0.002, i, f'{v:.4f}', va='center')

# Prédit vs Réel
y_pred_best = results_reg[best_reg_name]['pred']
idx = np.random.choice(len(y_te_r), 1000)
axes[1].scatter(y_te_r[idx], y_pred_best[idx], alpha=0.4, s=12, color='steelblue')
mn, mx = y_te_r.min(), y_te_r.max()
axes[1].plot([mn, mx], [mn, mx], 'r--', lw=2)
axes[1].set_xlabel('Puissance Réelle (kW)')
axes[1].set_ylabel('Puissance Prédite (kW)')
axes[1].set_title(f'🎯 Prédit vs Réel — {best_reg_name}', fontweight='bold')

plt.tight_layout()
plt.savefig('regression_puissance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Importance des features ───
rf_reg = results_reg['Random Forest']['model']
feat_imp = pd.Series(rf_reg.feature_importances_, index=features_reg).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors_imp = plt.cm.RdYlGn(feat_imp.values / feat_imp.max())
ax.barh(feat_imp.index, feat_imp.values, color=colors_imp)
ax.set_xlabel('Importance (Gini)')
ax.set_title('🔍 Importance des Variables — Prédiction Puissance Chauffage', fontweight='bold')
for i, v in enumerate(feat_imp.values):
    ax.text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('feature_importance_panache.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Modèle 3 : Classifieur Multi-classe des Décisions IA

In [ ]:
le = LabelEncoder()
y_dec = le.fit_transform(df['decision_ia'])

X_dec = df[features_cls].values
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(
    X_dec, y_dec, test_size=0.2, random_state=42, stratify=y_dec
)

clf_dec = RandomForestClassifier(
    n_estimators=200, max_depth=15, n_jobs=-1,
    random_state=42, class_weight='balanced'
)
clf_dec.fit(X_tr_d, y_tr_d)
y_pred_d = clf_dec.predict(X_te_d)

acc_dec = accuracy_score(y_te_d, y_pred_d)
print(f'🎯 Accuracy classifieur décisions : {acc_dec:.4f} ({acc_dec*100:.2f}%)')
print()
print(classification_report(y_te_d, y_pred_d, target_names=le.classes_))

# Matrice de confusion
fig, ax = plt.subplots(figsize=(9, 7))
cm_dec = confusion_matrix(y_te_d, y_pred_d)
sns.heatmap(cm_dec, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_xlabel('Prédiction')
ax.set_ylabel('Réalité')
ax.set_title('📊 Matrice Confusion — Classifieur Décisions IA', fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_decisions.png', dpi=150, bbox_inches='tight')
plt.show()

## 💰 6. Analyse Économique et Énergétique

In [ ]:
# ─── Analyse mensuelle ───
df_monthly = df.resample('M').agg({
    'puissance_ia_kW'       : 'mean',
    'puissance_sans_ia_kW'  : 'mean',
    'cout_ia_dth'           : 'sum',
    'cout_sans_ia_dth'      : 'sum',
    'economie_dth'          : 'sum',
    'panache_visible'       : 'mean',
}).reset_index()

df_monthly['economie_pct'] = (
    (df_monthly['cout_sans_ia_dth'] - df_monthly['cout_ia_dth'])
    / df_monthly['cout_sans_ia_dth'] * 100
)

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Coûts mensuels comparés
x = np.arange(len(df_monthly))
axes[0,0].fill_between(x, df_monthly['cout_sans_ia_dth'], alpha=0.4,
                        color='red', label='Sans IA')
axes[0,0].fill_between(x, df_monthly['cout_ia_dth'], alpha=0.7,
                        color='green', label='Avec IA')
axes[0,0].set_title('💰 Coût Mensuel Chauffage (DT)', fontweight='bold')
axes[0,0].set_ylabel('Coût (DT/mois)')
axes[0,0].legend()

# Économie mensuelle
bar_cols = ['#2ecc71' if v > 0 else '#e74c3c' for v in df_monthly['economie_dth']]
axes[0,1].bar(x, df_monthly['economie_dth'], color=bar_cols, alpha=0.85)
axes[0,1].axhline(df_monthly['economie_dth'].mean(), color='blue', linestyle='--', alpha=0.7,
                   label=f'Moy: {df_monthly["economie_dth"].mean():.0f} DT')
axes[0,1].set_title('💵 Économie Mensuelle IA (DT)', fontweight='bold')
axes[0,1].set_ylabel('Économie (DT)')
axes[0,1].legend()

# % économie mensuelle
axes[1,0].plot(x, df_monthly['economie_pct'], 'g-o', linewidth=2, markersize=5)
axes[1,0].fill_between(x, df_monthly['economie_pct'], alpha=0.2, color='green')
axes[1,0].axhline(df_monthly['economie_pct'].mean(), color='blue', linestyle='--', alpha=0.7,
                   label=f'Moy: {df_monthly["economie_pct"].mean():.1f}%')
axes[1,0].set_title('📉 % Réduction Coût Chauffage', fontweight='bold')
axes[1,0].set_ylabel('Économie (%)')
axes[1,0].set_ylim(0, 100)
axes[1,0].legend()

# Visibilité panache résiduelle
axes[1,1].bar(x, df_monthly['panache_visible'] * 100,
               color=plt.cm.RdYlGn_r(df_monthly['panache_visible'].values), alpha=0.85)
axes[1,1].set_title('👁️ Visibilité Résiduelle Panache (%)', fontweight='bold')
axes[1,1].set_ylabel('Panache visible (%)')
axes[1,1].axhline(df_monthly['panache_visible'].mean() * 100, color='red',
                   linestyle='--', label=f'Moy: {df_monthly["panache_visible"].mean()*100:.1f}%')
axes[1,1].legend()

plt.tight_layout()
plt.savefig('analyse_economique_panache.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\n💡 Bilan Économique sur 3 ans :')
print(f'   Coût SANS IA (continu)  : {df["cout_sans_ia_dth"].sum():,.0f} DT')
print(f'   Coût AVEC IA            : {df["cout_ia_dth"].sum():,.0f} DT')
print(f'   Économie totale         : {df["economie_dth"].sum():,.0f} DT')
print(f'   Économie annuelle       : {df["economie_dth"].sum()/3:,.0f} DT/an')
print(f'   % réduction moyenne     : {df["economie_dth"].sum()/df["cout_sans_ia_dth"].sum()*100:.1f}%')

## 🚦 7. Système de Décision Temps Réel

In [ ]:
class PanacheControlSystem:
    """
    Système IA de contrôle du chauffage des gaz
    pour éliminer la pollution visuelle des cheminées.
    """
    def __init__(self, clf_visible, reg_power, clf_decision,
                 scaler_cls, scaler_reg, le_dec, features_cls, features_reg):
        self.clf_visible   = clf_visible
        self.reg_power     = reg_power
        self.clf_decision  = clf_decision
        self.scaler_cls    = scaler_cls
        self.scaler_reg    = scaler_reg
        self.le_dec        = le_dec
        self.features_cls  = features_cls
        self.features_reg  = features_reg
        self.tarif_kwh     = 0.18  # DT/kWh

    def analyze(self, data: dict) -> dict:
        X_c = np.array([[data[f] for f in self.features_cls]])
        X_r = np.array([[data[f] for f in self.features_reg]])

        # Prédiction visibilité
        visible = self.clf_visible.predict(X_c)[0]
        prob_vis = self.clf_visible.predict_proba(X_c)[0][1]

        # Prédiction puissance optimale
        if data['heure'] >= 6 and data['heure'] <= 22:
            power = max(0, self.reg_power.predict(X_r)[0])
        else:
            power = 0.0

        # Décision
        dec_encoded = self.clf_decision.predict(X_c)[0]
        decision    = self.le_dec.inverse_transform([dec_encoded])[0]

        # Économie vs chauffage continu
        power_continu = 120 if 6 <= data['heure'] <= 22 else 60
        economie_h    = (power_continu - power) * self.tarif_kwh

        label_map = {
            'nuit_inactif'      : ('🌙', 'Nuit — Système inactif', 'Aucune action requise'),
            'chauffage_inutile' : ('✅', 'Panache naturellement invisible', 'Désactiver le chauffage'),
            'chauffage_leger'   : ('🟡', 'Faible risque de panache', f'Chauffage léger : {power:.0f} kW'),
            'chauffage_modere'  : ('🟠', 'Risque modéré de panache', f'Chauffage modéré : {power:.0f} kW'),
            'chauffage_maximal' : ('🔴', 'Fort risque — conditions humides', f'Chauffage maximal : {power:.0f} kW'),
        }
        emoji, statut, action = label_map.get(decision, ('❓', decision, ''))

        return {
            'panache_visible'   : bool(visible),
            'prob_visible_pct'  : round(prob_vis * 100, 1),
            'decision'          : decision,
            'emoji'             : emoji,
            'statut'            : statut,
            'action'            : action,
            'puissance_kW'      : round(power, 1),
            'puissance_continue': power_continu,
            'economie_h_DT'     : round(economie_h, 3),
        }

    def print_result(self, r):
        print(f"{'─'*60}")
        print(f"{r['emoji']}  {r['statut']}")
        print(f"   Panache visible    : {'OUI' if r['panache_visible'] else 'NON'} "
              f"(prob. {r['prob_visible_pct']}%)")
        print(f"   Décision IA        : {r['decision']}")
        print(f"   Action             : {r['action']}")
        print(f"   Puissance IA       : {r['puissance_kW']} kW  "
              f"(vs {r['puissance_continue']} kW continu)")
        print(f"   Économie horaire   : {r['economie_h_DT']} DT")
        print(f"{'─'*60}")


# Instancier
ctrl = PanacheControlSystem(
    clf_visible=results_cls[best_cls_name]['model'],
    reg_power=results_reg[best_reg_name]['model'],
    clf_decision=clf_dec,
    scaler_cls=scaler,
    scaler_reg=scaler_r,
    le_dec=le,
    features_cls=features_cls,
    features_reg=features_reg,
)

print('✅ Système de contrôle IA initialisé!\n')
print('🔄 Test sur 4 scénarios réels :')

scenarios = [
    {'label': 'Matin hivernal humide',
     'T_amb_C':10,'humidite_rel_pct':88,'T_rosee_C':8,
     'vitesse_vent_ms':3.2,'pression_hpa':1018,
     'T_gaz_procede_C':60,'H_gaz_pct':92,'debit_gaz_nm3h':42000,
     'delta_T_gaz_amb':50,'heure':9,'mois':1,'jour_semaine':1,'production_factor':0.9},

    {'label': 'Après-midi estival sec',
     'T_amb_C':35,'humidite_rel_pct':32,'T_rosee_C':16,
     'vitesse_vent_ms':5.0,'pression_hpa':1010,
     'T_gaz_procede_C':58,'H_gaz_pct':85,'debit_gaz_nm3h':48000,
     'delta_T_gaz_amb':23,'heure':15,'mois':7,'jour_semaine':2,'production_factor':1.0},

    {'label': 'Nuit (hors heures visibilité)',
     'T_amb_C':18,'humidite_rel_pct':75,'T_rosee_C':14,
     'vitesse_vent_ms':2.0,'pression_hpa':1012,
     'T_gaz_procede_C':57,'H_gaz_pct':90,'debit_gaz_nm3h':35000,
     'delta_T_gaz_amb':39,'heure':3,'mois':4,'jour_semaine':3,'production_factor':0.7},

    {'label': 'Printemps — conditions modérées',
     'T_amb_C':20,'humidite_rel_pct':65,'T_rosee_C':13,
     'vitesse_vent_ms':4.5,'pression_hpa':1015,
     'T_gaz_procede_C':62,'H_gaz_pct':88,'debit_gaz_nm3h':44000,
     'delta_T_gaz_amb':42,'heure':11,'mois':4,'jour_semaine':0,'production_factor':0.95},
]

for s in scenarios:
    label = s.pop('label')
    print(f'\n🏭 Scénario : {label}')
    result = ctrl.analyze(s)
    ctrl.print_result(result)

## 📊 8. Tableau de Bord Interactif (Plotly)

In [ ]:
df_week = df['2021-12-01':'2021-12-14']

fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[
        'Puissance IA vs Continue (kW)',
        'Humidité & Température ambiante',
        'Économie horaire cumulée (DT)',
        'Probabilité panache visible (%)',
        'Distribution décisions IA',
        'Économie mensuelle (DT)'
    ],
    vertical_spacing=0.12
)

fig.add_trace(go.Scatter(x=df_week.index, y=df_week['puissance_sans_ia_kW'],
    name='Sans IA', fill='tozeroy', fillcolor='rgba(231,76,60,0.15)',
    line=dict(color='#e74c3c')), row=1, col=1)
fig.add_trace(go.Scatter(x=df_week.index, y=df_week['puissance_ia_kW'],
    name='Avec IA', fill='tozeroy', fillcolor='rgba(46,204,113,0.3)',
    line=dict(color='#2ecc71')), row=1, col=1)

fig.add_trace(go.Scatter(x=df_week.index, y=df_week['humidite_rel_pct'],
    name='Humidité (%)', line=dict(color='#3498db')), row=1, col=2)
fig.add_trace(go.Scatter(x=df_week.index, y=df_week['T_amb_C'],
    name='Temp (°C)', line=dict(color='#e67e22'), yaxis='y4'), row=1, col=2)

eco_cum = df_week['economie_dth'].cumsum()
fig.add_trace(go.Scatter(x=df_week.index, y=eco_cum,
    name='Économie cumul.', fill='tozeroy', fillcolor='rgba(46,204,113,0.2)',
    line=dict(color='#27ae60')), row=2, col=1)

fig.add_trace(go.Scatter(x=df_week.index, y=df_week['panache_visible']*100,
    name='Panache (%)', line=dict(color='#9b59b6')), row=2, col=2)

dec_counts = df['decision_ia'].value_counts()
fig.add_trace(go.Pie(labels=dec_counts.index, values=dec_counts.values,
    hole=0.35, name='Décisions'), row=3, col=1)

monthly_eco = df.resample('M')['economie_dth'].sum().reset_index()
fig.add_trace(go.Bar(x=monthly_eco['timestamp'].astype(str),
    y=monthly_eco['economie_dth'], name='Éco/mois',
    marker_color='#2ecc71'), row=3, col=2)

fig.update_layout(
    height=900,
    title_text='🏭 Tableau de Bord IA — Contrôle Panache Visuel Cheminées',
    title_font_size=16,
    showlegend=False,
    template='plotly_white'
)
fig.show()
print('✅ Tableau de bord interactif généré!')

## 💾 9. Sauvegarde

In [ ]:
import os

df.to_csv('dataset_panache_visuel.csv')
print(f'✅ Dataset exporté ({os.path.getsize("dataset_panache_visuel.csv")/1024:.0f} KB)')

models_save = {
    'clf_visibilite_panache'  : results_cls[best_cls_name]['model'],
    'reg_puissance_chauffage' : results_reg[best_reg_name]['model'],
    'clf_decision_ia'         : clf_dec,
    'scaler_classification'   : scaler,
    'label_encoder_decision'  : le,
}
for name, obj in models_save.items():
    with open(f'{name}.pkl', 'wb') as f:
        pickle.dump(obj, f)
    print(f'   ✓ {name}.pkl sauvegardé')

print('\n✅ Tous les modèles sauvegardés!')

## 📋 10. Rapport Final

In [ ]:
acc_best   = results_cls[best_cls_name]['acc']
r2_best    = results_reg[best_reg_name]['r2']
eco_total  = df['economie_dth'].sum()
eco_an     = eco_total / 3
pct_eco    = eco_total / df['cout_sans_ia_dth'].sum() * 100
pct_invisible = (1 - df['panache_visible'].mean()) * 100
h_no_heat  = (df['puissance_ia_kW'] == 0).sum()

print('='*65)
print(' RAPPORT FINAL — IA CONTRÔLE PANACHE VISUEL CHEMINÉES')
print('='*65)
print(f'''
📁 DATASET GÉNÉRÉ
   ▸ 26 280 heures (3 ans, horaire)
   ▸ 24 variables météo + opératoires
   ▸ Localisation : Gabès, Tunisie

🤖 MODÈLES ML
   ▸ Prédiction visibilité  : Accuracy = {acc_best:.4f} ({best_cls_name})
   ▸ Prédiction puissance   : R² = {r2_best:.4f} ({best_reg_name})
   ▸ Classifieur décisions  : Accuracy = {acc_dec:.4f}

💰 BILAN ÉCONOMIQUE
   ▸ Économie totale (3 ans) : {eco_total:,.0f} DT
   ▸ Économie annuelle       : {eco_an:,.0f} DT/an
   ▸ % réduction coût        : {pct_eco:.1f}%
   ▸ Heures sans chauffage   : {h_no_heat:,} h ({h_no_heat/len(df)*100:.1f}%)

👁️  IMPACT VISUEL
   ▸ Panache invisible       : {pct_invisible:.1f}% du temps
   ▸ Période critique        : hivers matins (6h–10h)

🎯 VALEUR AJOUTÉE DE L'IA
   ▸ Décision temps réel selon conditions météo
   ▸ Économie énergie maximale (chauffage minimal)
   ▸ Acceptabilité sociale améliorée
   ▸ Aucune modification composition des gaz
   ▸ Déployable avec moyens propres
''')
print('='*65)
print('  ✅ Notebook complet — Solution IA Panache Visuel')
print('='*65)